# Collect DiD-BCF non-main metrics

Upload the metrics ZIPs from this non-main campaign, or add their paths in `INPUT_PATHS`. The collector validates assignments, fit identities, checksums, and the 8 registered reused fit(s). It reports missing jobs by phase and saves partial tables when phases remain incomplete. It does not read or include the separate main rerun's results. Empirical application ZIPs produced by the local assignment can be supplied here as well.


In [ ]:
import json, os, sys, subprocess, time
from pathlib import Path
assert (3, 12) <= sys.version_info[:2] <= (3, 14), "Use Python 3.12 to 3.14"
SESSION_STARTED = time.time()
DEADLINE = SESSION_STARTED + 9.5 * 3600
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
SOURCE_COMMIT = '30b165b27610dc4b2c406a9d64281d77e660257a'
REPO = Path("/content/unique_nonmain") if Path("/content").exists() else Path.cwd() / "unique_nonmain"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/hugogobato/DiD-BCF.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", SOURCE_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", 'Simulation_Studies_Revision/DiD_BCF/Nonmain_Rerun_20261014'], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == SOURCE_COMMIT, (actual, SOURCE_COMMIT)
PACKAGE = REPO / 'Simulation_Studies_Revision/DiD_BCF/Nonmain_Rerun_20261014'
manifest = json.loads((PACKAGE / "manifest.json").read_text())
requirements = [name + "==" + version for name, version in manifest["dependencies"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                *requirements], check=True)
HELPERS = PACKAGE / "helpers"
print("Pinned non-main sources and dependencies installed:", SOURCE_COMMIT)


In [ ]:
from google.colab import files
uploaded = files.upload()
UPLOADED_PATHS = [str(Path(name).resolve()) for name in uploaded if name.lower().endswith(".zip")]
INPUT_PATHS = []  # Optional paths such as "/content/didbcf_nonmain_local_application.zip".
INPUTS = list(dict.fromkeys([*UPLOADED_PATHS, *map(str, INPUT_PATHS)]))
if not INPUTS:
    raise ValueError("Upload at least one didbcf_nonmain_*.zip or add a ZIP path to INPUT_PATHS")
RESULTS = Path("/content/didbcf_nonmain_tables")
subprocess.run([sys.executable, str(HELPERS / "collect.py"), *INPUTS,
                "--out", str(RESULTS)], check=True)
status = json.loads((RESULTS / "status.json").read_text())
print("Phase expected:", status["phase_expected"])
print("Phase completed:", status["phase_completed"])
print("Missing fits:", status["missing_total"])
import shutil
output_file = shutil.make_archive("/content/didbcf_nonmain_tables", "zip", RESULTS)
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
